In [ ]:
import aspose.words_foss as aw 

docs_path = './docs/'
filename = "uk-rf.doc"

doc = aw.Document(docs_path+filename)
text = doc.get_text()

In [ ]:
import numpy as np

def split_into_chunks_fixed(text: str, chunk_size: int,  overlap: int=0) -> list[str]:
    # для каждого чанка выведем начальный и конечный индексы, чтобы 
    if overlap >= chunk_size:
        raise ValueError("chunk_size должен быть больше overlap!")

    if overlap < 0:
        raise ValueError("overlap не должен быть меньше нуля!")

    if chunk_size < 0:
        raise ValueError("chunk_size не может быть меньше нуля!")

    if not text:
        raise ValueError("Текст пустой!")

    start_indexes = np.arange(0, len(text), chunk_size-overlap)
    end_indexes = start_indexes + chunk_size

    chunks = [
        text[start:end] for start, end in zip(start_indexes, end_indexes)
    ]

    return chunks

In [ ]:
import torch
import polars as pl
import hashlib
from transformers import AutoTokenizer, AutoModel

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

cache_dir = './models/'
embeding_model_name = "google/embeddinggemma-2"

vdb_path = './database/'
vdb_filename = 'test.parquet'

tokenizer = AutoTokenizer.from_pretrained(embeding_model_name, cache_dir=cache_dir)
model = AutoModel.from_pretrained(embeding_model_name, cache_dir=cache_dir).to(device).eval()

def get_ids(chunks, tokenizer):
    tokens = [tokenizer.tokenize(chunk) for chunk in chunks]

    ids = [
        tokenizer.convert_tokens_to_ids(x)
        for x in tokens
    ]

    prepared_ids = tokenizer.pad(
        {"input_ids": ids},
        padding=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    return prepared_ids

def get_embeddings(batch, model):
    with torch.inference_mode():
        output = model(**batch)

    token_embeddings = output.last_hidden_state
    token_embeddings_prepared = (
        batch['attention_mask']
        .to(device)
        .to(torch.float64)
        .unsqueeze(-1)
    ) * (
        token_embeddings
    )

    return token_embeddings_prepared.mean(dim=1)

def write_base(text, model, tokenizer, batch_size=4000, vdb_path='./database/', vdb_filename='base.parquet'):
    batch_size = 4000
    overlap = 40
    text_len = len(text)

    writer = None

    for step in np.arange(0, text_len, batch_size):
        current_text = text[step:step + batch_size]
        chunks = split_into_chunks_fixed(current_text, batch_size, overlap)

        ids = get_ids(chunks, tokenizer)

        batch = {
            key: value.to(device)
            for key, value in ids.items()
        }

        embedding_vectors = get_embeddings(batch, model)

        table = pl.DataFrame({
            "id": [hashlib.sha256(chunk.encode('utf-8')).hexdigest() for chunk in chunks],
            "text": chunks,
            "embedding_vector": embedding_vectors.cpu().float().numpy().tolist()
        })

        arrow_table = table.to_arrow()

        if writer is None:
            writer = pq.ParquetWriter(
                vdb_path+vdb_filename,
                arrow_table.schema,
                compression="zstd",
            )

        writer.write_table(arrow_table)

    if writer is not None:
        writer.close()

write_base(text, model, tokenizer)

In [ ]:
# search in base
query1 = "Мне нужна узнать: могу ли я своровать и быть безнаказанным"
query2 = "В какой статье есть наказание за убийство?"
query3 = "Какой кодекс в арбитражном суде?"

def search(query, model, tokenizer, base_file):
    # tokenizer

    # embedding_vector = get_embeddings()
    pass

In [33]:
import torch.nn.functional as F
tokens = tokenizer.tokenize(query1)
ids = tokenizer.convert_tokens_to_ids(tokens)
batch = tokenizer.pad(
    {'input_ids': [ids]},
    padding=True,
    return_attention_mask=True,
    return_tensors='pt'    
)
model.cpu()
output = F.normalize(
    model(**batch)
    .last_hidden_state
    .squeeze()
    .mean(dim=1),
    dim=0    
)

In [34]:
output

tensor([ 0.0859, -0.0156, -0.3867, -0.7969, -0.2441, -0.0244, -0.0255, -0.1514,
        -0.0442, -0.3418, -0.0342, -0.0059, -0.0159, -0.0300,  0.0623],
       dtype=torch.bfloat16)

In [ ]:
def cosine_similarity(x, y):
    return x @ y

def retrieval(batches, query_embeddings):
    for batch in batches:
        score = cosine_similarity(batch, query_embeddings)
        